In [5]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Generate a robust retail transaction dataset locally
np.random.seed(42)
n_rows = 1500

categories = {
    'Technology': ['Phones', 'Accessories', 'Laptops'],
    'Furniture': ['Chairs', 'Tables', 'Bookcases'],
    'Office Supplies': ['Storage', 'Paper', 'Binders']
}

cat_list = np.random.choice(list(categories.keys()), n_rows, p=[0.3, 0.3, 0.4])
subcat_list = [np.random.choice(categories[c]) for c in cat_list]
regions = np.random.choice(['North', 'South', 'East', 'West'], n_rows)
segments = np.random.choice(['Consumer', 'Corporate', 'Home Office'], n_rows, p=[0.5, 0.3, 0.2])

sales = np.round(np.random.exponential(scale=150, size=n_rows) + 20, 2)
quantities = np.random.randint(1, 10, size=n_rows)
# Margin simulation: Technology tends to be profitable, Furniture has tighter/negative margins
margin_rates = [0.25 if c == 'Technology' else (-0.05 if c == 'Furniture' else 0.15) for c in cat_list]
profits = np.round(sales * (margin_rates + np.random.normal(0, 0.1, n_rows)), 2)

dates = pd.date_range(start='2025-01-01', periods=n_rows, freq='4h')

df = pd.DataFrame({
    'Order_ID': [f"ORD-{1000 + i}" for i in range(n_rows)],
    'Order_Date': dates.strftime('%Y-%m-%d'),
    'Region': regions,
    'Segment': segments,
    'Category': cat_list,
    'Sub_Category': subcat_list,
    'Sales': sales,
    'Quantity': quantities,
    'Profit': profits
})

# 2. Spin up the in-memory SQLite database
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

print(f"Dataset successfully created: {df.shape[0]} rows, {df.shape[1]} columns.\n")
print("Columns ready for SQL & Python querying:")
print(list(df.columns))

Dataset successfully created: 1500 rows, 9 columns.

Columns ready for SQL & Python querying:
['Order_ID', 'Order_Date', 'Region', 'Segment', 'Category', 'Sub_Category', 'Sales', 'Quantity', 'Profit']


In [6]:
# ==========================================
# 1. THE SQL APPROACH
# ==========================================
# In SQL:
# SELECT -> What columns or calculations you want to view
# FROM   -> The table name ('orders')
# GROUP BY -> The dimension you want to summarize by (like Pivot Rows)
# ORDER BY -> How you want to sort the results (DESC = descending / highest first)

sql_query = """
SELECT
    Category,
    COUNT(Order_ID) AS Total_Orders,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct
FROM orders
GROUP BY Category
ORDER BY Total_Sales DESC;
"""

sql_result = pd.read_sql(sql_query, conn)
print("--- [1] SQL OUTPUT (Pivot by Category) ---")
display(sql_result)

# ==========================================
# 2. THE PYTHON (PANDAS) APPROACH
# ==========================================
# In Pandas:
# .groupby('Category') -> Does what GROUP BY does in SQL / Rows in Pivot
# .agg(...)            -> Aggregates: sum, count, mean, etc.
# .sort_values(...)    -> Does what ORDER BY does in SQL / Sort A-Z in Excel

py_result = df.groupby('Category').agg(
    Total_Orders=('Order_ID', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

# Calculating Margin Column: [Profit / Sales * 100]
py_result['Profit_Margin_Pct'] = (py_result['Total_Profit'] / py_result['Total_Sales'] * 100).round(2)
py_result['Total_Sales'] = py_result['Total_Sales'].round(2)
py_result['Total_Profit'] = py_result['Total_Profit'].round(2)

# Sort highest sales first
py_result = py_result.sort_values(by='Total_Sales', ascending=False)

print("\n--- [2] PYTHON PANDAS OUTPUT (Exact Same Result) ---")
display(py_result)

--- [1] SQL OUTPUT (Pivot by Category) ---


,Category,Total_Orders,Total_Sales,Total_Profit,Profit_Margin_Pct
0,Office Supplies,613,99812.39,14208.19,14.23
1,Technology,466,83239.59,21208.62,25.48
2,Furniture,421,69655.46,-4253.24,-6.11



--- [2] PYTHON PANDAS OUTPUT (Exact Same Result) ---


,Category,Total_Orders,Total_Sales,Total_Profit,Profit_Margin_Pct
1,Office Supplies,613,99812.39,14208.19,14.23
2,Technology,466,83239.59,21208.62,25.48
0,Furniture,421,69655.46,-4253.24,-6.11


In [7]:
# ==========================================
# 1. SQL DRILL-DOWN (WHERE Clause)
# ==========================================
# WHERE Category = 'Furniture' -> Filters before grouping
# GROUP BY Sub_Category        -> Breaks down the problem by sub-item

sql_furniture_query = """
SELECT
    Sub_Category,
    COUNT(Order_ID) AS Total_Orders,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Margin_Pct
FROM orders
WHERE Category = 'Furniture'
GROUP BY Sub_Category
ORDER BY Margin_Pct ASC;
"""

furniture_sql_df = pd.read_sql(sql_furniture_query, conn)
print("--- [SQL] FURNITURE SUB-CATEGORY BREAKDOWN ---")
display(furniture_sql_df)

# ==========================================
# 2. PYTHON (PANDAS) DRILL-DOWN
# ==========================================
# df['Category'] == 'Furniture' -> The filter condition

furniture_py_df = (
    df[df['Category'] == 'Furniture']
    .groupby('Sub_Category')
    .agg(
        Total_Orders=('Order_ID', 'count'),
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum')
    )
    .reset_index()
)

furniture_py_df['Margin_Pct'] = (furniture_py_df['Profit'] / furniture_py_df['Sales'] * 100).round(2)
furniture_py_df = furniture_py_df.sort_values(by='Margin_Pct', ascending=True)

print("\n--- [PYTHON] MATCHING SUB-CATEGORY BREAKDOWN ---")
display(furniture_py_df)

--- [SQL] FURNITURE SUB-CATEGORY BREAKDOWN ---


,Sub_Category,Total_Orders,Sales,Profit,Margin_Pct
0,Chairs,137,24996.12,-1831.09,-7.33
1,Bookcases,133,20219.33,-1388.29,-6.87
2,Tables,151,24440.01,-1033.86,-4.23



--- [PYTHON] MATCHING SUB-CATEGORY BREAKDOWN ---


,Sub_Category,Total_Orders,Sales,Profit,Margin_Pct
1,Chairs,137,24996.12,-1831.09,-7.33
0,Bookcases,133,20219.33,-1388.29,-6.87
2,Tables,151,24440.01,-1033.86,-4.23


In [8]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, Reference
from google.colab import files

# 1. Create workbook & sheets
wb = openpyxl.Workbook()
ws_dash = wb.active
ws_dash.title = "Executive Dashboard"
ws_data = wb.create_sheet(title="Raw Data")

# Turn off grid lines look or keep clean
ws_dash.views.sheetView[0].showGridLines = True

# 2. Populate Raw Data sheet
for r in dataframe_to_rows(df, index=False, header=True):
    ws_data.append(r)

# 3. Design Executive Dashboard
# Title Banner
ws_dash.merge_cells("B2:J3")
title_cell = ws_dash["B2"]
title_cell.value = "EXECUTIVE RETAIL PERFORMANCE & PROFITABILITY DASHBOARD"
title_cell.font = Font(name="Segoe UI", size=15, bold=True, color="FFFFFF")
title_cell.fill = PatternFill(start_color="1E293B", end_color="1E293B", fill_type="solid")
title_cell.alignment = Alignment(horizontal="center", vertical="center")

# KPI 1: Total Sales
ws_dash["B5"] = "Total Sales"
ws_dash["B6"] = f"${df['Sales'].sum():,.2f}"
# KPI 2: Total Profit
ws_dash["D5"] = "Total Profit"
ws_dash["D6"] = f"${df['Profit'].sum():,.2f}"
# KPI 3: Overall Margin
overall_margin = (df['Profit'].sum() / df['Sales'].sum()) * 100
ws_dash["F5"] = "Profit Margin"
ws_dash["F6"] = f"{overall_margin:.2f}%"

# Format KPI Cards
for col_letter in ["B", "D", "F"]:
    lbl = ws_dash[f"{col_letter}5"]
    val = ws_dash[f"{col_letter}6"]
    lbl.font = Font(name="Segoe UI", size=9, color="64748B", bold=True)
    lbl.fill = PatternFill(start_color="F8FAFC", fill_type="solid")
    val.font = Font(name="Segoe UI", size=14, color="0F172A", bold=True)
    val.fill = PatternFill(start_color="F8FAFC", fill_type="solid")
    lbl.alignment = Alignment(horizontal="center")
    val.alignment = Alignment(horizontal="center")

# 4. Insert Aggregated Category Performance Table
summary_cat = df.groupby('Category').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()

ws_dash["B9"] = "Category"
ws_dash["C9"] = "Total Sales"
ws_dash["D9"] = "Total Profit"

for cell in ["B9", "C9", "D9"]:
    ws_dash[cell].font = Font(name="Segoe UI", bold=True, color="FFFFFF")
    ws_dash[cell].fill = PatternFill(start_color="334155", fill_type="solid")
    ws_dash[cell].alignment = Alignment(horizontal="center")

row_idx = 10
for _, r in summary_cat.iterrows():
    ws_dash[f"B{row_idx}"] = r['Category']
    ws_dash[f"C{row_idx}"] = round(r['Sales'], 2)
    ws_dash[f"D{row_idx}"] = round(r['Profit'], 2)
    ws_dash[f"C{row_idx}"].number_format = '$#,##0.00'
    ws_dash[f"D{row_idx}"].number_format = '$#,##0.00'
    row_idx += 1

# 5. Insert an Automated Native Excel Bar Chart
chart = BarChart()
chart.type = "col"
chart.style = 10
chart.title = "Sales vs Profit by Category"
chart.y_axis.title = "USD ($)"
chart.x_axis.title = "Category"
chart.height = 10
chart.width = 15

data_ref = Reference(ws_dash, min_col=3, min_row=9, max_col=4, max_row=row_idx-1)
cats_ref = Reference(ws_dash, min_col=2, min_row=10, max_row=row_idx-1)
chart.add_data(data_ref, titles_from_data=True)
chart.set_categories(cats_ref)

ws_dash.add_chart(chart, "F9")

# Auto-fit column widths
for ws in [ws_dash, ws_data]:
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = openpyxl.utils.get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

# Save & Trigger Auto-Download
filename = "Retail_Executive_Dashboard.xlsx"
wb.save(filename)
print(f"Generated {filename}! Downloading to your PC...")
files.download(filename)

Generated Retail_Executive_Dashboard.xlsx! Downloading to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# ==============================================================================
# SQL WINDOW FUNCTIONS: DENSE_RANK() and SUM() OVER(PARTITION BY ...)
# ==============================================================================
# 1. PARTITION BY Category: Resets the calculation for every product category.
# 2. DENSE_RANK() OVER(...): Ranks sub-categories by sales within their own category.
# 3. SUM(Sales) OVER(PARTITION BY Category): Calculates the category total on every row,
#    allowing you to compute exact % contribution of each product line.

window_sql = """
WITH Product_Summaries AS (
    SELECT
        Category,
        Sub_Category,
        ROUND(SUM(Sales), 2) AS Subcat_Sales,
        ROUND(SUM(Profit), 2) AS Subcat_Profit
    FROM orders
    GROUP BY Category, Sub_Category
)
SELECT
    Category,
    Sub_Category,
    Subcat_Sales,
    Subcat_Profit,
    -- Rank products inside their respective category
    DENSE_RANK() OVER(
        PARTITION BY Category
        ORDER BY Subcat_Sales DESC
    ) AS Rank_In_Category,
    -- Category sales total preserved on every line
    ROUND(SUM(Subcat_Sales) OVER(PARTITION BY Category), 2) AS Category_Total_Sales,
    -- Percentage contribution of this item to its category
    ROUND(Subcat_Sales * 100.0 / SUM(Subcat_Sales) OVER(PARTITION BY Category), 2) AS Pct_Of_Category
FROM Product_Summaries
ORDER BY Category, Rank_In_Category;
"""

window_result = pd.read_sql(window_sql, conn)
display(window_result)

,Category,Sub_Category,Subcat_Sales,Subcat_Profit,Rank_In_Category,Category_Total_Sales,Pct_Of_Category
0,Furniture,Chairs,24996.12,-1831.09,1,69655.46,35.89
1,Furniture,Tables,24440.01,-1033.86,2,69655.46,35.09
2,Furniture,Bookcases,20219.33,-1388.29,3,69655.46,29.03
3,Office Supplies,Storage,35860.27,4538.23,1,99812.39,35.93
4,Office Supplies,Binders,33660.88,5014.09,2,99812.39,33.72
5,Office Supplies,Paper,30291.24,4655.87,3,99812.39,30.35
6,Technology,Phones,33046.40,8734.83,1,83239.59,39.70
7,Technology,Accessories,25981.94,7105.73,2,83239.59,31.21
8,Technology,Laptops,24211.25,5368.06,3,83239.59,29.09
